# 🎓 Guía didáctica: API del Actor `pdf-text-extractor` (Apify)

Este notebook explica, paso a paso y con ejemplos ejecutables, cómo usar los endpoints de la API de Apify para el Actor **`automation-lab/pdf-text-extractor`**, usando la librería `requests` de Python.

## ⚠️ Sobre el token

Todas las URLs de la API de Apify llevan un parámetro `?token=...`. Ese token es tu credencial personal:

- **Nunca lo compartas** ni lo subas a un repositorio público.
- En este notebook lo vamos a cargar como una variable/secreto, **nunca hardcodeado en el código**.
- En Colab, lo ideal es guardarlo en `🔑 Secrets` (panel lateral izquierdo) con el nombre `APIFY_TOKEN`.

## 🗺️ Qué vamos a ver

1. Conceptos: qué es un Actor, un run, un dataset y un key-value store en Apify
2. Configuración inicial (token, imports)
3. Ejecutar el Actor de forma asíncrona (`run`)
4. Ejecutar el Actor de forma síncrona (`run-sync`)
5. Ejecutar y obtener directamente los ítems del dataset (`run-sync-get-dataset-items`)
6. Consultar información del Actor (`GET actor`)
7. Listar versiones del Actor
8. Listar y crear webhooks
9. Actualizar configuración del Actor y de una versión
10. Listar builds y construir una versión
11. Listar runs y obtener el último run
12. Obtener los ítems del dataset del último run
13. Obtener la definición OpenAPI del Actor
14. Eliminar el Actor (con cuidado ⚠️)

## 1. Conceptos clave

| Concepto | Qué es |
|---|---|
| **Actor** | Un programa/scraper empaquetado que corre en la infraestructura de Apify (acá: `automation-lab/pdf-text-extractor`) |
| **Run** | Una ejecución concreta de un Actor, con su propio ID, estado (`READY`, `RUNNING`, `SUCCEEDED`, `FAILED`...) |
| **Build** | Una versión compilada/lista para ejecutar del código del Actor |
| **Dataset** | Almacenamiento de resultados en forma de lista de ítems (uno por fila, tipo tabla) |
| **Key-value store** | Almacenamiento de pares clave-valor (por ejemplo, el registro `OUTPUT` de un run) |
| **Webhook** | Notificación HTTP automática que Apify dispara cuando pasa algo (ej: cuando un run termina) |

Todos los endpoints de este notebook siguen el mismo patrón base:

```
https://api.apify.com/v2/actors/{usuario}~{nombre-actor}/{recurso}?token={tu_token}
```

In [ ]:
%pip install apify-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.5/150.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.6 MB/s eta 0:00:00


In [ ]:
from apify_client import ApifyClient

# Initialize the ApifyClient with your API token
client = ApifyClient("<YOUR_API_TOKEN>")

# Prepare the Actor input
run_input = {
    "urls": [
        "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf",
    ],
    "maxConcurrency": 5,
    "timeoutPerPdfSecs": 60,
    "includePages": True,
}

# Run the Actor and wait for it to finish
run = client.actor("xxxxx").call(run_input=run_input)

# Fetch and print Actor results from the run's dataset (if there are any)
for item in client.dataset(run["defaultDatasetId"]).iterate_items():
    print(item)

## 2. Configuración inicial

Cargamos el token de forma segura (Colab Secrets) y armamos la URL base.

In [ ]:
import requests

# --- Opción A: Colab Secrets (recomendado) ---
# Panel izquierdo -> icono de llave 🔑 -> "Add new secret" -> nombre: APIFY_TOKEN
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get("APIFY_TOKEN")
except Exception:
    # --- Opción B: pedirlo de forma interactiva (no queda guardado en el notebook) ---
    import getpass
    APIFY_TOKEN = getpass.getpass("Pegá tu token de Apify: ")

ACTOR = "automation-lab~pdf-text-extractor"
BASE_URL = f"https://api.apify.com/v2/actors/{ACTOR}"

print("Base URL configurada:", BASE_URL)

## 3. Ejecutar el Actor de forma asíncrona

**Endpoint:** `POST /runs`

Inicia el Actor y devuelve **inmediatamente** información del run (todavía en estado `READY` o `RUNNING`). No espera a que termine. Es ideal para procesos largos donde después consultás el estado por separado.

In [ ]:
# El INPUT depende de lo que espere el Actor. Para un extractor de texto de PDF,
# algo típico sería la URL del PDF a procesar.
input_data = {
    "pdfUrl": "https://example.com/documento.pdf"
}

response = requests.post(
    f"{BASE_URL}/runs",
    params={"token": APIFY_TOKEN},
    json=input_data
)

run_info = response.json()
print("Status HTTP:", response.status_code)
print("ID del run:", run_info["data"]["id"])
print("Estado:", run_info["data"]["status"])

## 4. Ejecutar de forma síncrona y obtener el resultado directo

**Endpoint:** `POST /run-sync`

Corre el Actor y **espera** a que termine, devolviendo el registro `OUTPUT` del key-value store por defecto. Útil para tareas rápidas. Ojo: si la conexión HTTP es muy larga, puede cortarse.

In [ ]:
response = requests.post(
    f"{BASE_URL}/run-sync",
    params={"token": APIFY_TOKEN},
    json=input_data
)

print("Status HTTP:", response.status_code)
print("OUTPUT devuelto:")
print(response.text[:500])

## 5. Ejecutar de forma síncrona y obtener los ítems del dataset

**Endpoint:** `POST /run-sync-get-dataset-items`

Igual que el anterior, pero en vez de devolver el `OUTPUT` del key-value store, devuelve directamente los ítems que el Actor haya guardado en su **dataset** (por ejemplo, el texto extraído de cada página).

In [ ]:
response = requests.post(
    f"{BASE_URL}/run-sync-get-dataset-items",
    params={"token": APIFY_TOKEN},
    json=input_data
)

items = response.json()
print(f"Se recibieron {len(items)} ítems del dataset.")
if items:
    print("Primer ítem:", items[0])

## 6. Obtener la configuración del Actor

**Endpoint:** `GET /` (sin sufijo)

Devuelve metadatos del Actor: nombre, descripción, versiones, autor, etc.

In [ ]:
response = requests.get(BASE_URL, params={"token": APIFY_TOKEN})
actor_data = response.json()["data"]

print("Nombre:", actor_data["name"])
print("ID:", actor_data["id"])
print("Última modificación:", actor_data.get("modifiedAt"))

## 7. Listar versiones del Actor

**Endpoint:** `GET /versions`

In [ ]:
response = requests.get(f"{BASE_URL}/versions", params={"token": APIFY_TOKEN})
versiones = response.json()["data"]["items"]

for v in versiones:
    print(f"- Versión {v['versionNumber']} (buildTag: {v.get('buildTag', 'N/A')})")

## 8. Listar webhooks del Actor

**Endpoint:** `GET /webhooks`

Un webhook es una URL propia que Apify llama automáticamente ante eventos (ej: `ACTOR.RUN.SUCCEEDED`).

In [ ]:
response = requests.get(f"{BASE_URL}/webhooks", params={"token": APIFY_TOKEN})
webhooks = response.json()["data"]["items"]

print(f"Cantidad de webhooks configurados: {len(webhooks)}")
for w in webhooks:
    print("-", w.get("eventTypes"), "->", w.get("requestUrl"))

## 9. Actualizar la configuración del Actor y de una versión

**Endpoints:** `PUT /` y `PUT /versions/{versionNumber}`

Se usa un `PUT` con un JSON parcial: solo los campos que querés modificar.

In [ ]:
# Ejemplo: actualizar la descripción del Actor (¡cuidado, esto modifica datos reales!)
cambios_actor = {"description": "Extractor de texto de archivos PDF - actualizado desde notebook"}

# Descomentar para ejecutar de verdad:
# response = requests.put(BASE_URL, params={"token": APIFY_TOKEN}, json=cambios_actor)
# print(response.status_code, response.json())

print("Ejemplo de payload para PUT /actor:", cambios_actor)

# Actualizar una versión específica (ej: 0.0)
cambios_version = {"buildTag": "latest"}

# response = requests.put(
#     f"{BASE_URL}/versions/0.0",
#     params={"token": APIFY_TOKEN},
#     json=cambios_version
# )
print("Ejemplo de payload para PUT /versions/0.0:", cambios_version)

## 10. Listar builds y construir una nueva versión

**Endpoints:** `GET /builds` y `POST /builds?version=0.0`

Un *build* compila el código del Actor para que quede listo para ejecutarse.

In [ ]:
# Listar builds existentes
response = requests.get(f"{BASE_URL}/builds", params={"token": APIFY_TOKEN})
builds = response.json()["data"]["items"]
print(f"Cantidad de builds: {len(builds)}")

# Disparar un nuevo build de la versión 0.0
response = requests.post(
    f"{BASE_URL}/builds",
    params={"token": APIFY_TOKEN, "version": "0.0"}
)
print("Nuevo build iniciado:", response.json()["data"]["id"])

## 11. Listar runs y obtener el último run

**Endpoints:** `GET /runs` y `GET /runs/last`

Tip: agregando `?status=SUCCEEDED` a `/runs/last`, obtenés el último run que haya terminado con éxito.

In [ ]:
# Todos los runs
response = requests.get(f"{BASE_URL}/runs", params={"token": APIFY_TOKEN})
runs = response.json()["data"]["items"]
print(f"Cantidad total de runs: {len(runs)}")

# Último run exitoso
response = requests.get(
    f"{BASE_URL}/runs/last",
    params={"token": APIFY_TOKEN, "status": "SUCCEEDED"}
)
if response.status_code == 200:
    ultimo_run = response.json()["data"]
    print("Último run exitoso:", ultimo_run["id"], "| terminado:", ultimo_run.get("finishedAt"))
else:
    print("Todavía no hay runs exitosos.")

## 12. Obtener los ítems del dataset del último run

**Endpoint:** `GET /runs/last/dataset/items`

Este endpoint acepta además todos los parámetros del endpoint general "Dataset Get Items" (por ejemplo `format`, `limit`, `offset`, `fields`, etc.).

In [ ]:
response = requests.get(
    f"{BASE_URL}/runs/last/dataset/items",
    params={
        "token": APIFY_TOKEN,
        "status": "SUCCEEDED",
        "format": "json",
        "limit": 10
    }
)

if response.status_code == 200:
    items = response.json()
    print(f"Ítems obtenidos: {len(items)}")
else:
    print("No se pudo obtener el dataset del último run (status:", response.status_code, ")")

## 13. Obtener la definición OpenAPI del Actor

**Endpoint:** `GET /builds/default/openapi.json`

Muy útil: te da un esquema OpenAPI/Swagger completo, ideal para generar clientes automáticos o documentación interactiva.

In [ ]:
response = requests.get(f"{BASE_URL}/builds/default/openapi.json")
openapi_spec = response.json()

print("Título:", openapi_spec.get("info", {}).get("title"))
print("Versión:", openapi_spec.get("info", {}).get("version"))
print("Paths disponibles:", list(openapi_spec.get("paths", {}).keys()))

## 14. Eliminar el Actor ⚠️

**Endpoint:** `DELETE /`

**Acción destructiva e irreversible**: borra el Actor y todos sus datos asociados (versiones, builds, runs, datasets). Lo dejamos comentado a propósito.

In [ ]:
# ⚠️ DESCOMENTAR SOLO SI REALMENTE QUERÉS BORRAR EL ACTOR
# response = requests.delete(BASE_URL, params={"token": APIFY_TOKEN})
# print("Status:", response.status_code)

print("Celda de seguridad: no se ejecutó ningún DELETE.")

## ✅ Resumen de endpoints

| Método | Endpoint | Qué hace |
|---|---|---|
| POST | `/runs` | Ejecuta el Actor (asíncrono) |
| POST | `/run-sync` | Ejecuta y espera, devuelve el `OUTPUT` |
| POST | `/run-sync-get-dataset-items` | Ejecuta y espera, devuelve ítems del dataset |
| GET | `/` | Info del Actor |
| PUT | `/` | Actualiza el Actor |
| DELETE | `/` | Elimina el Actor |
| GET | `/versions` | Lista versiones |
| PUT | `/versions/{n}` | Actualiza una versión |
| GET | `/webhooks` | Lista webhooks |
| GET | `/builds` | Lista builds |
| POST | `/builds?version=n` | Crea un build nuevo |
| GET | `/runs` | Lista runs |
| GET | `/runs/last` | Último run |
| GET | `/runs/last/dataset/items` | Ítems del dataset del último run |
| GET | `/builds/default/openapi.json` | Definición OpenAPI |

## 🔒 Buenas prácticas de seguridad

- Nunca pegues tu token directamente en el código si vas a compartir el notebook.
- Usá `google.colab.userdata` (Secrets) o variables de entorno.
- Revocá y regenerá el token si sospechás que quedó expuesto (Apify Console → Settings → Integrations).
- Las operaciones `PUT`/`DELETE` modifican datos reales: probalas primero contra un Actor de prueba.

## 📚 Recursos

- Documentación general de la API de Apify: https://docs.apify.com/api/v2
- Documentación de `requests`: https://requests.readthedocs.io/